# nano Beatrix — the readout tie: the request read off where the trained head looks

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_readout.ipynb)

**The question**: Alhena (Castor's body turned by the anchored multi-position rotary) answers a question exactly when it is handed the address it asks for, and dozens of independently trained arms then merge under the gate with nothing lost. The address has to come from the model itself. The last two screens (Alhena-P, `colab/nano_pointer.ipynb`; Alhena-P2, `colab/nano_pointer2.ipynb`) told one attention head (block 4, head 0) where the record is at each index entry. On Alhena-P that head does look at the right record: the record it attends most is the asked one on at least 98% of the entries on 5 of 8 runs. The index the model writes is that right on 2 of the 8. Other paths in the network write the index, and its second digit is often copied from a neighbouring record.

**The readout tie** reads the address off the head's look instead of off the written bytes: the address of the record holding more than half of the head's attention at the index entry, both digits taken from that one record. When no record holds more than half (a split look), there is no request and every arm abstains:

    M(r) = the head's attention summed over record r's bytes;    request = address(r) if M(r) > 1/2, else none

No model is trained here. The look is read on the stored models of both screens, and at the same head of the Alhena-V models as a control (those were trained without the signal). Alhena-P seed 2's stored arms are read through the look again. The scale test's arms are then trained, each alone, on up to five models whose look passes the rule, and merged through it.

| part | what | why |
|---|---|---|
| the look read | each stored model's index as requests: the written index; the look at the index entry, on the row as composed and on the model's own free-running row; the look at the question itself; the answers through each; 512 read rows and 128 key-hard rows | is the look an exact index where the written one is not |
| read 0 | Alhena-P seed 2's stored arms, read through the look | the arms merged through the look on a stored set |
| members | the scale test's 48 real arms + 48 toy members (after the last block) and 16 real arms (after block 3), each trained alone, on up to five models whose look on the free-running row is right on at least .98 of the questions and whose answers with the true address reach .95 | merged under w_i = sinh(u_i) / (1 + sum_j (cosh(u_j) - 1)), the look as the request |

**The cells**: C1 pins the trainer and the arm library and holds the decision block; C2 builds the data (the arms' training text and the held-out rows), fetches the 23 stored models, prices the look read on this card and probes the push path; C3 reads the look on the 23 models; C4 applies the member rule; C5 reads Alhena-P seed 2's stored arms through the look; C6 trains the members on the models the rule picked; C7 renders the tables. Every record ships to [AbstractPhil/alephllm-nano-beatrix-training](https://huggingface.co/AbstractPhil/alephllm-nano-beatrix-training) (`bed/reports/look/`, `bed/reports/read0/`, `bed/scale_members/`).

**Before C1**: the notebook's `HF_TOKEN` secret (write access to that repo) with notebook access on. Run C1, then Runtime > Restart session and run all. A file named `STOP` in `/content/out` ends the session between models; a restarted session resumes where it stopped. The code and its tests: [AbstractEyes/geolip-nano-beatrix-trainer](https://github.com/AbstractEyes/geolip-nano-beatrix-trainer) (`geolip/nano_beatrix/rotary_scale.py`: `look_of`, `look_read`).

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART the session after the first run of this cell (Runtime > Restart session and run all).
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.80" "amoe-lora @ git+https://github.com/AbstractEyes/amoe-lora@v0.2.12"
import torch, amoe
import geolip.nano_beatrix as N
print('geolip.nano_beatrix', N.__version__, '| amoe', amoe.__version__, '| torch', torch.__version__, '| card',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 80), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart session, then rerun from C1')
assert amoe.__version__ == '0.2.12', 'the arm module of record is amoe-lora 0.2.12'
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN            # the runner and the shipper read it from the environment; the value is never printed

# ============ THE DECISION BLOCK — every setting set from the plan, with its source beside it ============
OUT = '/content/out'                                   # the session folder: the data, the models, the records, the members
REPO = 'AbstractPhil/alephllm-nano-beatrix-training'  # every record ships here; tokenless = nothing ships
CORPUS_BYTES = 160e6                                   # fineweb-edu text: the arms' training rows (the scale test's corpus) and the held-out rows
TAIL = '_L24000_ctx4096_frdyn_cap_mode_sz32-128-256-512_r4_n30_o0_askany_gag50_cp4_va1'
_tag = lambda s, piece, series: f'alhena-both_s{s}{TAIL}{piece}_ahdigit1_{series}'
LOOK_HEAD = (4, 0)                                     # block 4 (numbered from 1), head 0: the head both screens' signal trained
CORES = ([(f'P s{s}', _tag(s, '_pt10b4h0', 'pointer'), s, 'ptr') for s in (1, 2, 3, 4, 5, 6)]        # Alhena-P at .1: their own head
         + [('P at .3 s1', _tag(1, '_pt30b4h0', 'pointer'), 1, 'ptr')]
         + [(f'P s{s}', _tag(s, '_pt10b4h0', 'pointerrep'), s, 'ptr') for s in (7, 8)]
         + [(f'P2 s{s}', _tag(s, '_pt10b4h0p2sg', 'pointer2'), s, 'ptr') for s in (1, 2, 3, 4, 5, 6)]
         + [(f'V s{s}', _tag(s, '', 'value'), s, LOOK_HEAD) for s in (1, 2, 3, 4, 5, 6)]              # the control: the same head,
         + [(f'V s{s}', _tag(s, '', 'valuerep'), s, LOOK_HEAD) for s in (7, 8)])                      # trained without the signal
P_S2 = CORES[1][1]                                     # Alhena-P seed 2: its stored arms (bed/scale_members/<tag>_members_l16000_m32000/)
MIN_LOOK, MIN_ANSWERS = 0.98, 0.95                     # the member rule: the look (free-running) right >= .98, the answers with the true address >= .95
N_MEMBER_CORES = 5                                     # at most five new member cores, the look's right highest first; never a control
LAST_STEPS, MID_STEPS = 16000, 32000                   # the scale test's member budget, verbatim
READ_BATCHES, CALIB_BATCHES, HARD_BATCHES = 64, 32, 16 # 512 read rows, 256 calibration rows, 128 key-hard rows (8 rows a batch)
PRICE_STEPS = 200                                      # the member price lines (each trainer on throwaway stacks)
PLAN_MIN_LOOK = 1.5                                    # the planned minutes of one model's look read on this card (an estimate)
PLAN_MIN_MEM = 18.8                                    # the planned training minutes of one core's members (4.9 last + 13.8 mid measured)
STOP_FACTOR = 1.5                                      # a projection past 1.5x the plan stops the session for a re-plan
print('decision block:', len(CORES), 'models | the member rule', MIN_LOOK, MIN_ANSWERS, 'at most', N_MEMBER_CORES, '| out', OUT,
      '| ships to', REPO if HF_TOKEN else 'nothing (tokenless)')

In [ ]:
# C2 (P) — PREFLIGHT: the data, the 23 stored models, the look read priced on this card, the push probe. Nothing trains.
import json, os, shutil, time, torch
from huggingface_hub import snapshot_download
from geolip.nano_beatrix import bed_corpus
from geolip.nano_beatrix import rotary_scale as rs
if not os.path.exists(f'{OUT}/bed/bed_corpus.bin'):
    bed_corpus.build(f'{OUT}/bed', CORPUS_BYTES)
HELD = f'{OUT}/bed/heldout_fineweb_edu_head2048.jsonl'
print('corpus', os.path.getsize(f'{OUT}/bed/bed_corpus.bin') / 1e6, 'MB; held-out rows', os.path.exists(HELD))
HUB = f'{OUT}/hub'
t0 = time.time()
snapshot_download(repo_id=REPO, local_dir=HUB, allow_patterns=[f'bed/weights/{t}/*' for _, t, _, _ in CORES])
FOLDER = {t: f'{HUB}/bed/weights/{t}' for _, t, _, _ in CORES}
missing = [n for n, t, _, _ in CORES if not os.path.exists(f'{FOLDER[t]}/model.safetensors')]
assert not missing, f'models missing on the Hub: {missing}'
print(f'{len(CORES)} models fetched in {time.time() - t0:.0f} s')
rs.look_read(FOLDER[CORES[0][1]], f'{OUT}/price/look_warm.json', CORES[0][2], HELD, look=CORES[0][3], read_batches=1, hard_batches=0,
             log=lambda s: None)                       # the card's first calls (kernels, caches) kept out of the price
t0 = time.time()                                       # the price line: the first model again, 4 read batches and 1 key-hard batch
rs.look_read(FOLDER[CORES[0][1]], f'{OUT}/price/look_price.json', CORES[0][2], HELD, look=CORES[0][3], read_batches=4, hard_batches=1,
             log=lambda s: None)
per = (time.time() - t0) * (READ_BATCHES + HARD_BATCHES) / 5 / 60
PRICE_OK = per <= STOP_FACTOR * PLAN_MIN_LOOK
print(f'this card: one look read about {per:.1f} min (planned {PLAN_MIN_LOOK}; the model load included); the {len(CORES)} models '
      f'about {per * len(CORES):.0f} min' + ('' if PRICE_OK else f' -> PAST {STOP_FACTOR}x THE PLAN: C3 will not start'))
PUSH_OK = False
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    path = 'bed/preflight/push_probe_readout.json'
    body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'card': torch.cuda.get_device_name(0),
                       'look_read_min': round(per, 2), 'models': len(CORES)}).encode()
    api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message='nano readout: push probe')
    PUSH_OK = path in set(api.list_repo_files(REPO))
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
else:
    print('TOKENLESS: no probe written; the cells below run and ship nothing')

def ship_file(path, dst_dir, ship):
    # one record to dst_dir on the Hub: staged alone in its own folder, so the upload carries only it
    st = f'{OUT}/stage/{os.path.basename(path)[:-5]}'
    os.makedirs(st, exist_ok=True)
    shutil.copy(path, st)
    ship.submit(st, dst_dir)

In [ ]:
# C3 (T) — THE LOOK READ on the 23 models, one at a time, resume-first (a record in OUT/look/ is not read again); a STOP file in OUT starts
# no new model; each record ships to bed/reports/look/ as it lands (one background shipper: the card never waits on the Hub).
assert PRICE_OK, 'the look read priced past the plan: re-plan before C3'
ship = rs.Shipper(REPO) if (HF_TOKEN and PUSH_OK) else None
t_all = time.time()
for n, (label, tag, seed, look) in enumerate(CORES, 1):
    if os.path.exists(f'{OUT}/STOP'):
        print('STOP present: no new model starts')
        break
    out = f'{OUT}/look/{rs.look_name(tag, LOOK_HEAD)}.json'
    t0 = time.time()
    rec = rs.look_read(FOLDER[tag], out, seed, HELD, look=look, read_batches=READ_BATCHES, hard_batches=HARD_BATCHES, log=lambda s: None)
    assert (rec['look']['block'], rec['look']['head']) == LOOK_HEAD, (label, rec['look'])
    if ship:
        ship_file(out, 'bed/reports/look', ship)
    q = rec['read']['request']
    print(f"{label}: the look at the index entry, free-running {q['LOOKF']['right']:.4f} (composed row {q['LOOK']['right']:.4f}); the "
          f"written index {q['INDEXF']['right']:.4f}; the look at the question {q['LOOKQF']['right']:.4f}; {(time.time() - t0) / 60:.1f} min; "
          f"{(time.time() - t_all) / 60:.1f} min spent; {len(CORES) - n} models left")
if ship:
    print('ships drained' if ship.drain() else 'ships still retrying in the background')

In [ ]:
# C4 — THE MEMBER RULE (the plan's, written before any read): every stored P and P2 model but P s2 (its arms are stored) whose look on the
# free-running row is right on >= .98 of the read's questions and whose answers with the true address reach .95, the look's right highest
# first, at most five; never a control (V).
cand = []
for label, tag, seed, look in CORES:
    if label.startswith('V') or tag == P_S2:
        continue
    p = f'{OUT}/look/{rs.look_name(tag, LOOK_HEAD)}.json'
    if not os.path.exists(p):
        print(f'{label}: no look record in this session')
        continue
    rec = json.load(open(p, encoding='utf-8'))
    lf, ans = rec['read']['request']['LOOKF']['right'], rec['read']['answers']['LAYOUT']
    ok = lf >= MIN_LOOK and ans >= MIN_ANSWERS
    print(f'{label}: the look (free-running) {lf:.4f}, the answers with the true address {ans:.4f} -> ' + ('a member core' if ok else 'not'))
    if ok:
        cand.append((lf, label, tag, seed))
cand.sort(key=lambda c: -c[0])                         # the look's right, highest first (ties keep the models' order)
MEMBER_CORES = [(label, tag, seed) for _, label, tag, seed in cand[:N_MEMBER_CORES]]
print('member cores, in order:', [c[0] for c in MEMBER_CORES] or 'none')

In [ ]:
# C5 (T) — READ 0: Alhena-P seed 2's stored arms read again in the two-pass form with the look sources beside the stored ones (the
# continuity check against its stored read); resume-first; the record ships to bed/reports/read0/.
cf, mf = rs.fetch_stored(REPO, P_S2, f'{P_S2}_members_l16000_m32000', HUB)
out0 = f'{OUT}/read0/{P_S2}_read0_look.json'
t0 = time.time()
rec0 = rs.read0(cf, mf, out0, 2, HELD, read_batches=READ_BATCHES, calib_batches=CALIB_BATCHES, forms=('find',))
print('the stored cells re-read:', rec0.get('continuity_find'), f'| {(time.time() - t0) / 60:.1f} min')
if HF_TOKEN and PUSH_OK:
    s0 = rs.Shipper(REPO)
    ship_file(out0, 'bed/reports/read0', s0)
    print('shipped' if s0.drain() else 'the ship still retrying in the background')

In [ ]:
# C6 (T) — THE MEMBERS on C4's member cores, in its order: the scale test's recipe verbatim (48 real arms + 48 toy members after block 6,
# 16 real arms after block 3, each trained alone on the frozen model), read in the two-pass form with the look sources; one core at a time,
# resume-first; each core's members ship to bed/scale_members/ as they land.
def run_members(cores):
    owned, never = rs.owned_sets()
    packed = rs.load_packed(f'{OUT}/bed')
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    core = rs.Core(FOLDER[cores[0][1]], 'cuda')
    cc = rs.compile_check(core, packed, owned, never)
    print('compiled = eager at step 1 (largest difference over the largest value):', cc)
    assert all(v['write'] < 1e-4 and v['grad'] < 1e-3 for v in cc.values()), 'the compiled writes are not the eager writes: no member trains'
    pr = rs.price(core, packed, owned, never, steps=PRICE_STEPS)
    per_core = (pr['last_ms_per_step'] * LAST_STEPS + pr['mid_ms_per_step'] * MID_STEPS) / 60000
    print(f"this card: last site {pr['last_ms_per_step']:.1f} ms a step, mid site {pr['mid_ms_per_step']:.1f} ms a step -> a core's members "
          f"about {per_core:.0f} min of training (planned {PLAN_MIN_MEM}) + the reads")
    assert per_core <= STOP_FACTOR * PLAN_MIN_MEM, 'the members priced past the plan: re-plan before any member trains'
    del core
    torch.cuda.empty_cache()
    ship = rs.Shipper(REPO) if (HF_TOKEN and PUSH_OK) else None
    t_all = time.time()
    out = []
    for n, (label, tag, seed) in enumerate(cores, 1):
        if os.path.exists(f'{OUT}/STOP'):
            print('STOP present: no new core starts')
            break
        print(f'=================== members {n}/{len(cores)} {label} ===================')
        t0 = time.time()
        res = rs.run_core(FOLDER[tag], f'{OUT}/members', seed, f'{OUT}/bed', HELD, last_steps=LAST_STEPS, mid_steps=MID_STEPS,
                          read_batches=READ_BATCHES, calib_batches=CALIB_BATCHES, shipper=ship, forms=('find',))
        out.append(res)
        torch.cuda.empty_cache()
        print(f'{label}: {(time.time() - t0) / 60:.1f} min; {(time.time() - t_all) / 60:.1f} min spent; {len(cores) - n} cores left')
    if ship:
        print('ships drained' if ship.drain() else 'ships still retrying in the background')
    return out

done_m = run_members(MEMBER_CORES) if MEMBER_CORES else []
print('members:', [r['name'] for r in done_m] or 'none (no model passed the rule)')

In [ ]:
# C7 — THE TABLES: the look read (per model: the requests' right, the look's no-request and wrong-record shares, the answers through each,
# the key-hard rows) and the members (per core and stack: solo against bare, the nine rest cells keeping all three marks and QUIET per
# source, the controls, the largest leak through the look, the mid site, the text bits). Run any time.
import glob, json
from IPython.display import Markdown, display
rows = ['| model | written index (free-running) | look: composed / free-running / at the question | free-running look: none / wrong record | '
        'answers: true / written / look / look at the question | look mass (decoded) / split / duplicate | key-hard rows: written / look | min |',
        '|---|---|---|---|---|---|---|---|']
for label, tag, seed, look in CORES:
    p = f'{OUT}/look/{rs.look_name(tag, LOOK_HEAD)}.json'
    if not os.path.exists(p):
        continue
    r = json.load(open(p, encoding='utf-8'))
    q, a, lk = r['read']['request'], r['read']['answers'], r['read']['look']['LOOKF']
    h = r.get('hard', {}).get('request', {})
    hard = f"{h['INDEXF']['right']:.4f} / {h['LOOKF']['right']:.4f}" if h else '-'
    rows.append(f"| {label} | {q['INDEXF']['right']:.4f} | {q['LOOK']['right']:.4f} / {q['LOOKF']['right']:.4f} / {q['LOOKQF']['right']:.4f} | "
                f"{lk['no_request']:.4f} / {lk['wrong_record']:.4f} | {a['LAYOUT']:.4f} / {a['INDEXF']:.4f} / {a['LOOKF']:.4f} / {a['LOOKQF']:.4f} | "
                f"{lk['decoded_mass_mean']:.3f} / {lk['split']:.4f} / {lk['duplicate']:.4f} | {hard} | {r['wall_s'] / 60:.1f} |")
display(Markdown('\n'.join(rows)))
allm = lambda x: all(x[k] for k in ('PROTECTED', 'QUIET', 'EXACT'))
def _nine(c, s, mark=None):
    if f'{s}_U5_REST_m16' not in c:
        return '-'
    ks = [f'{s}_U{u}_REST_m{m}' for u in (5, 8, 10) for m in (16, 32, 48)]
    return f"{sum(allm(c[k]) if mark is None else c[k][mark] for k in ks)}/9"
rows = ['| core | stack | solo / bare owned | look free-running: all three / QUIET | written: all three / QUIET | true address | look composed | '
        'look at the question | G2 m48 | INDEXC m48 Q | G0 m48 P/Q | largest leak / own (look) | mid look U5/8/10 | G0 text bits |',
        '|---|---|---|---|---|---|---|---|---|---|---|---|---|---|']
reads = [(f'{d}/last_find.json', f'{d}/mid_find.json') for d in sorted(glob.glob(f'{OUT}/members/*_members_*'))]
reads += [(p, None) for p in sorted(glob.glob(f'{OUT}/read0/*_read0_look.json'))]
for lp, mp in reads:
    if not os.path.exists(lp):
        continue
    L = json.load(open(lp, encoding='utf-8'))
    rd = L['read'] if 'read' in L else L['find']
    M = json.load(open(mp, encoding='utf-8')) if mp and os.path.exists(mp) else None
    for st, r in rd['stacks'].items():
        c = r['cells']
        leak = (max(c[f'LOOKF_U{u}_REST_m{m}']['leak_over_own_write'] for u in (5, 8, 10) for m in (16, 32, 48))
                if 'LOOKF_U5_REST_m16' in c else float('nan'))
        mid = '-'
        if M and st == 'arms':
            mc = M['read']['cells']
            mid = ' '.join(str(allm(mc[f'LOOKF_U{u}_REST_m16'])) if f'LOOKF_U{u}_REST_m16' in mc else '-' for u in (5, 8, 10))
        g0 = (L.get('text_bits') or {}).get(f'G0_{st}_minus_bare')
        rows.append(f"| {L['core'][:30]}{' (read 0)' if 'find' in L else ''} | {st} | {r['solo']['ONE']['48']:.3f} / {rd['bare']['owned']:.3f} | "
                    f"{_nine(c, 'LOOKF')} / {_nine(c, 'LOOKF', 'QUIET')} | {_nine(c, 'INDEXF')} / {_nine(c, 'INDEXF', 'QUIET')} | "
                    f"{_nine(c, 'LAYOUT')} | {_nine(c, 'LOOK')} | {_nine(c, 'LOOKQF')} | {allm(c['G2_m48'])} | {c['INDEXC_U5_REST_m48']['QUIET']} | "
                    f"{c['G0_m48']['PROTECTED']}/{c['G0_m48']['QUIET']} | {leak:.1e} | {mid} | " + ('-' if g0 is None else f'{g0:+.3f}') + ' |')
display(Markdown('\n'.join(rows)))